In [3]:
import torch

print("Đã cài đặt thành công PyTorch phiên bản:", torch.__version__)
print("Apple Silicon (MPS) đã sẵn sàng chưa:", torch.backends.mps.is_available())

Đã cài đặt thành công PyTorch phiên bản: 2.8.0
Apple Silicon (MPS) đã sẵn sàng chưa: True


In [6]:
import os
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# 1. Định nghĩa Data Augmentation
IMG_SIZE = 224
BATCH_SIZE = 32

data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(), 
        transforms.RandomRotation(10),     
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]) 
    ]),
    'val': transforms.Compose([ 
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

# 2. Trỏ đúng tên thư mục là 'dataset'
data_dir = 'dataset' 

# 3. Tải dữ liệu lên
image_datasets = {
    x: datasets.ImageFolder(os.path.join(data_dir, x), data_transforms[x])
    for x in ['train', 'val'] 
}

# 4. Đóng gói thành các Batch
dataloaders = {
    x: DataLoader(image_datasets[x], batch_size=BATCH_SIZE, shuffle=True if x == 'train' else False)
    for x in ['train', 'val'] 
}

dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val']}
class_names = image_datasets['train'].classes

print(f"Các nhãn phân loại (Classes): {class_names}")
print(f"Số lượng ảnh Train: {dataset_sizes['train']}")
print(f"Số lượng ảnh Validation: {dataset_sizes['val']}")

Các nhãn phân loại (Classes): ['FAKE', 'REAL']
Số lượng ảnh Train: 20000
Số lượng ảnh Validation: 100000


In [8]:
import torch # Dòng quan trọng nhất tui quên mất đây nè 😂
import time
import copy
import torch.nn as nn
import torch.optim as optim
from torchvision import models

# Kiểm tra lại thiết bị (đảm bảo dùng Apple Silicon MPS)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

# 1. Tải mô hình ResNet-50 đã học trước trên ImageNet
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

# 2. Đóng băng tất cả các lớp (Feature Extraction)
for param in model.parameters():
    param.requires_grad = False

# 3. Chỉnh sửa lớp cuối cùng (Classification Head)
num_ftrs = model.fc.in_features
model.fc = nn.Sequential(
    nn.Linear(num_ftrs, 512),
    nn.ReLU(),
    nn.Dropout(0.4), # Tắt ngẫu nhiên 40% neuron để chống học vẹt (overfitting)
    nn.Linear(512, 2) # Chỉ có 2 kết quả: FAKE hoặc REAL
)

# Đưa mô hình vào GPU/MPS
model = model.to(device)

# 4. Cài đặt Hàm mất mát (Loss) và Thuật toán tối ưu (Optimizer)
criterion = nn.CrossEntropyLoss()
# Chỉ tối ưu hóa phần model.fc vừa mới thêm vào
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

print("Đã tải và cấu hình xong mô hình ResNet-50!")

1.0%

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /Users/tranbaonguyen/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100.0%


Đã tải và cấu hình xong mô hình ResNet-50!


In [13]:
import time
import copy
import torch

def train_model(model, criterion, optimizer, num_epochs=5):
    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')
        print('-' * 10)

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()  
            else:
                model.eval()   

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / dataset_sizes[phase]
            # Đã sửa lỗi Apple Silicon: Dùng .float() thay vì .double()
            epoch_acc = running_corrects.float() / dataset_sizes[phase]

            print(f'{phase.capitalize()} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())

        print()

    time_elapsed = time.time() - since
    print(f'Huấn luyện xong trong {time_elapsed // 60:.0f} phút {time_elapsed % 60:.0f} giây')
    print(f'Độ chính xác Validation tốt nhất: {best_acc:4f}')

    model.load_state_dict(best_model_wts)
    return model

In [14]:
# 1. Mở khóa và cài đặt lại tốc độ học (chạy lại cho chắc)
for param in model.layer4.parameters():
    param.requires_grad = True

optimizer_fine_tune = optim.Adam([
    {'params': model.layer4.parameters(), 'lr': 1e-5},
    {'params': model.fc.parameters(), 'lr': 1e-4}
])

print("Bắt đầu huấn luyện Fine-Tuning (Chờ máy chạy xong 5 Epochs nhé)...")

# 2. Huấn luyện (Chính dòng này tạo ra biến model_ft)
# Fen NHỚ CHỜ nó chạy xong hết 5/5 Epochs nhé!
model_ft = train_model(model, criterion, optimizer_fine_tune, num_epochs=5)

# 3. Tự động lưu mô hình ngay khi học xong
model_save_path = 'ai_image_detector.pth'
torch.save(model_ft.state_dict(), model_save_path)
print(f"🎉 Đã lưu mô hình thành công tại: {model_save_path}")

Bắt đầu huấn luyện Fine-Tuning (Chờ máy chạy xong 5 Epochs nhé)...
Epoch 1/5
----------
Train Loss: 0.1893 Acc: 0.9236
Val Loss: 0.1756 Acc: 0.9294

Epoch 2/5
----------
Train Loss: 0.1655 Acc: 0.9353
Val Loss: 0.1577 Acc: 0.9385

Epoch 3/5
----------
Train Loss: 0.1355 Acc: 0.9462
Val Loss: 0.1525 Acc: 0.9438

Epoch 4/5
----------
Train Loss: 0.1247 Acc: 0.9510
Val Loss: 0.1343 Acc: 0.9495

Epoch 5/5
----------
Train Loss: 0.1036 Acc: 0.9599
Val Loss: 0.1375 Acc: 0.9502

Huấn luyện xong trong 86 phút 36 giây
Độ chính xác Validation tốt nhất: 0.950210
🎉 Đã lưu mô hình thành công tại: ai_image_detector.pth
